In [1]:
from pathlib import Path
import mne
import numpy as np
from fooof import FOOOF

/var/folders/rh/p5yby44s3_q7gw0x4t4155p40000gq/T/ipykernel_90900/3248309884.py:4: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  from fooof import FOOOF


In [2]:
DATA_FOLDER1 = Path("DATA/posterior")
DATA_FOLDER2 = Path("DATA/frontal")

EPOCH_FILES1 = sorted(DATA_FOLDER1.glob("*-epo.fif"))
EPOCH_FILES2 = sorted(DATA_FOLDER2.glob("*-epo.fif"))

In [ ]:
def run_fooof(file_path, label):
    # Load preprocessed EEG epochs
    epochs = mne.read_epochs(file_path, preload=True)
    
    # Calculate PSD
    psd = epochs.compute_psd(
        method="welch",
        fmin=1,
        fmax=45,
        picks="eeg"
    )
    
    freqs = psd.freqs
    spectra = psd.get_data()  # epochs' PSD: channels x frequencies
    spectrum = spectra.mean(axis=0)  # average across EEG channels
    
    for ch_idx, ch_name in enumerate(psd.ch_names):
        fm = FOOOF(
            peak_width_limits=(1, 8),
            max_n_peaks=6,
            aperiodic_mode="fixed"
        )

        fm.report(freqs, spectrum[ch_idx], [3, 40])

        print(f"\n{label} | {file_path.name} | {ch_name}")
        print("Aperiodic parameters:", fm.aperiodic_params_)
        print("Peaks [center frequency, power, bandwidth]:")
        print(fm.peak_params_)
        print("R²:", fm.r_squared_)